# TinyLlama Chat Server with Cloudflare Tunnel

This notebook runs a FastAPI server that exposes a TinyLlama chat model via a Cloudflare tunnel. You can connect to it using the `cli.py` client.

In [ ]:
# Cell 0: Install dependencies and load the model
!pip install -q transformers accelerate torch fastapi uvicorn pycloudflared

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded ✅")

In [ ]:
# Cell 1: Start the server and tunnel
import uvicorn
from fastapi import FastAPI, Request
from pycloudflared import try_cloudflare
import threading
import time
import torch

app = FastAPI()

@app.post("/chat")
async def chat(request: Request):
    data = await request.json()
    message = data.get("message", "")
    print(f"Received message: {message}")
    
    # Generate response
    prompt = f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{message}</s>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    bot_response = full_response.split("<|assistant|>\n")[-1].strip()
    
    return {"response": bot_response}

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start background server
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

print("\nStarting Cloudflare tunnel...")
public_url = try_cloudflare(port=8000)
print(f"\nPublic URL: {public_url}")